# 🏦 FinTrust Machine Learning Engineering Track — Week 3 Interactive Walkthrough
## 🚀 Task: FinTrust ML Workflow — Integration-Ready Development
**Lead Mentor Teaching & Demonstration Notebook**

---

## 🎯 Session Learning Objectives for Interns
By the end of this interactive walkthrough, interns will understand:
1. **Part A (Workflow Audit)**: Why evaluating transactions in isolation fails and how to identify architecture gaps.
2. **Part B (Multi-Table Data Prep & PII Protection)**: Joining customer demographics with transactions while redacting sensitive PII (`Customer_Name`).
3. **Part B (Validation Firewall)**: Enforcing strict schema, type, range, and categorical contracts before inference.
4. **Part B (60-Feature Preprocessing)**: Leakage-free temporal extraction and one-hot encoding.
5. **Part C (The Model Adapter Pattern)**: Decoupling ML models from serving using `BaseModelAdapter` and `ExternalModelAdapter`.
6. **Part D (The 7-Stage Prediction Pipeline)**: Tracing data from ingestion through auto-enrichment to risk triage.
7. **Part D (30-Point Automated Quality Assurance)**: Running deterministic tests in `pytest`.
8. **Part E (FastAPI Microservice)**: Real-time REST API serving with Swagger UI.
9. **Part F (Reproducibility & Docker)**: Containerizing and diagnostic health checking.

--- 
## 1️⃣ Environment & Setup
First, we initialize the Python path and import our modular source code from `src/`.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import joblib

from src.config import PATHS, SCHEMA, MODEL_CONFIG
from src.data.prepare import DataPreparator
from src.validation.validator import DataValidator, DataValidationError
from src.preprocessing.pipeline import FinTrustDataPreprocessor, TemporalFeatureExtractor
from src.models.interface import BaseModelAdapter, FinTrustModelAdapter, ExternalModelAdapter
from src.models.predict import FinTrustPredictionPipeline, predict_single_transaction

print("✅ All Week 3 FinTrust modules loaded successfully!")
print(f"📁 Data Directory:     {PATHS.PROCESSED_DATA_DIR if hasattr(PATHS, 'PROCESSED_DATA_DIR') else PATHS.processed_enriched_path.parent}")
print(f"📁 Artifacts Directory: {PATHS.ARTIFACTS_DIR if hasattr(PATHS, 'ARTIFACTS_DIR') else PATHS.model_artifact_path.parent}")

--- 
## 2️⃣ Part B: Multi-Table Data Preparation & PII Redaction
### 💡 Mentor Teaching Note:
In Week 2, our model evaluated transactions without knowing the customer's age, income band, or digital profile.  
In Week 3, `DataPreparator`:
1. Loads 12,000 transactions and 1,500 customer profiles.
2. **Redacts `Customer_Name`** for NDPR/GDPR data privacy compliance.
3. Performs a relational left join on `Customer_ID` with a **100% match rate**.

In [ ]:
# Execute data preparation
preparator = DataPreparator()
prep_results = preparator.prepare_and_export()

print(f"✅ Total Enriched Records: {prep_results['total_enriched_records']:,}")
print(f"✅ Training Split:         {prep_results['train_records']:,} rows")
print(f"✅ Hold-out Test Split:    {prep_results['test_records']:,} rows")

# Inspect sample enriched records
df_enriched = pd.read_csv(PATHS.processed_enriched_path)
display(df_enriched[[
    "Transaction_ID", "Customer_ID", "Amount_NGN", "Transaction_Type",
    "Age", "Customer_Segment", "Monthly_Income_Band", "Digital_Engagement_Score", "Risk_Review_Flag"
]].head(5))

--- 
## 3️⃣ Part B: Data Validation Firewall
### 💡 Mentor Teaching Note:
Never let raw, unverified data reach your ML model!  
Let us test how `DataValidator` passes clean data and blocks malicious or corrupt inputs.

In [ ]:
# Test 1: Valid enriched dataset validation
validator = DataValidator(schema=SCHEMA, strict=False)
is_valid, report, _ = validator.validate(df_enriched, is_training=True, dataset_type="enriched")
report.print_report()

# Test 2: Testing malicious / corrupt input blocking
corrupt_input = pd.DataFrame([{
    "Transaction_ID": "FT-BAD-01",
    "Customer_ID": "FT-C00001",
    "Transaction_DateTime": 46023.0,
    "Transaction_Type": "Transfer",
    "Amount_NGN": -1000.0,            # ❌ Negative amount!
    "Channel": "Telepathy",           # ❌ Invalid category!
    "Device_Type": "Android",
    "Location": "Lagos",
    "International_Transaction": "No",
    "Transaction_Status": "Successful",
    "Age": 12.0,                      # ❌ Underage customer (<18)!
    "Tenure_Months": 10.0,
    "Digital_Engagement_Score": 50.0,
    "Gender": "Male",
    "Customer_Segment": "Everyday",
    "Account_Type": "Savings",
    "Monthly_Income_Band": "100k-249k",
    "Preferred_Channel": "Mobile App",
    "Account_Status": "Active"
}])

strict_val = DataValidator(schema=SCHEMA, strict=True)
try:
    strict_val.validate(corrupt_input, dataset_type="enriched")
except DataValidationError as e:
    print("\n🛡️ SECURITY FIREWALL BLOCKED CORRUPT INPUT:")
    for err in e.report.errors:
        print(f"   ❌ [{err.error_type}] {err.field}: {err.message}")

--- 
## 4️⃣ Part B: 60-Feature Preprocessing & Leakage Protection
### 💡 Mentor Teaching Note:
1. **Temporal Extraction:** Derives `Transaction_Hour`, `Transaction_DayOfWeek`, and `Is_Weekend` from timestamp serials.
2. **Unseen Category Safety:** Uses `OneHotEncoder(handle_unknown='ignore')` so unobserved categories at test time don't crash the server.
3. **Zero Data Leakage:** Scaling and imputation parameters are fitted strictly on `X_train`.

In [ ]:
# Load preprocessor
preprocessor = FinTrustDataPreprocessor.load(PATHS.preprocessor_artifact_path)

# Transform a sample of enriched records
sample_df = df_enriched.head(5)
transformed_matrix = preprocessor.transform(sample_df)

print(f"✅ Transformed feature matrix shape: {transformed_matrix.shape} (5 rows x 60 columns)")
print(f"✅ Total NaNs in transformed matrix: {np.isnan(transformed_matrix).sum()}")
print("\nSample Transformed Feature Vector (First Record, first 10 columns):")
print(transformed_matrix[0, :10])

--- 
## 5️⃣ Part C: Model Integration & The Adapter Pattern
### 💡 Mentor Teaching Note:
The **Adapter Pattern** decouples model development from the serving engine.  
Let us demonstrate loading our production `FinTrustModelAdapter` and even creating an `ExternalModelAdapter` to wrap an external Data Science model!

In [ ]:
# 1. Load production model adapter
model_adapter = FinTrustModelAdapter.load(PATHS.model_artifact_path)
metadata = model_adapter.get_metadata()

print("✅ Loaded FinTrust Production Model Adapter:")
print(f"   • Model Name:     {metadata.get('model_name')}")
print(f"   • Version:        {metadata.get('version')}")
print(f"   • Framework:      {metadata.get('model_family')}")
print(f"   • Risk Threshold: {model_adapter.classification_threshold}")

# 2. Demonstrate ExternalModelAdapter for peer Data Science models
from sklearn.linear_model import LogisticRegression
mock_peer_model = LogisticRegression()
mock_peer_model.fit(transformed_matrix, np.array([0, 1, 0, 1, 0]))

peer_adapter = ExternalModelAdapter(
    external_model=mock_peer_model,
    author_name="DS Intern Jane Doe",
    model_name="Peer Logistic Classifier",
    version="0.9.1"
)

peer_preds = peer_adapter.predict(transformed_matrix)
print(f"\n✅ Peer Adapter Predictions: {peer_preds.tolist()}")
print(f"   • Contributor: {peer_adapter.get_metadata()['author']}")

--- 
## 6️⃣ Part D: The 7-Stage End-to-End Prediction Pipeline
### 💡 Mentor Teaching Note:
Let us trace a single transaction from ingestion all the way to the banking operations decision.  
Notice how the pipeline **automatically looks up** customer demographics for `FT-C00001` from the customer feature store!

In [ ]:
pipeline = FinTrustPredictionPipeline(strict_validation=True)

# Suspicious nocturnal international transaction payload (only transaction attributes provided!)
suspicious_payload = {
    "Transaction_ID": "FT-T888001",
    "Customer_ID": "FT-C00001",        # Auto-enriches Age, Customer_Segment, Income Band
    "Transaction_DateTime": 46023.125,  # 03:00 AM
    "Transaction_Type": "Transfer",
    "Amount_NGN": 450000.0,            # High value transfer (NGN 450,000)
    "Channel": "Web",
    "Device_Type": "Web Browser",
    "Location": "Lagos",
    "International_Transaction": "Yes",
    "Transaction_Status": "Successful"
}

# Run through 7-stage workflow with verbose diagnostic logging
result_df = pipeline.run_pipeline(suspicious_payload, verbose=True)

print("\n🏦 FINAL STRUCTURED BANKING TRIAGE DECISION:")
display(result_df[[
    "Transaction_ID", "Customer_ID", "Age", "Customer_Segment",
    "Predicted_Risk_Flag", "Risk_Probability", "Risk_Tier", "Operational_Action"
]])

--- 
## 7️⃣ Part D: Automated 30-Point Technical Testing Matrix
### 💡 Mentor Teaching Note:
We can execute the automated `pytest` test suite directly from Python to show students how 30 tests pass.

In [ ]:
import subprocess

print("🧪 Running pytest test suite across 30 automated technical tests...\n")
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-v", "--tb=short"],
    cwd=project_root,
    capture_output=True,
    text=True
)
print(result.stdout)

--- 
## 8️⃣ Part E: FastAPI REST Microservice Testing
### 💡 Mentor Teaching Note:
We can use FastAPI's `TestClient` right here in the notebook to show interns how mobile banking apps interact with our API.

In [ ]:
from fastapi.testclient import TestClient
from src.api.app import app

client = TestClient(app)

# 1. Health check
res_health = client.get("/health")
print(f"GET /health -> HTTP {res_health.status_code}: {res_health.json()}")

# 2. Model metadata
res_meta = client.get("/model/metadata")
print(f"\nGET /model/metadata -> HTTP {res_meta.status_code}:")
print(json.dumps(res_meta.json(), indent=2))

# 3. Live transaction scoring
res_pred = client.post("/predict", json=suspicious_payload)
print(f"\nPOST /predict -> HTTP {res_pred.status_code}:")
print(json.dumps(res_pred.json(), indent=2))

--- 
## 9️⃣ Part F: One-Click Master System Diagnostic
### 💡 Mentor Teaching Note:
Run `check_all.py` at the end of the class to show that every single dependency, dataset, model, test, and deliverable is 100% healthy!

In [ ]:
from check_all import main as run_diagnostic
run_diagnostic()

---
## 🎉 Congratulations!
You have completed the **Week 3 Integration-Ready ML Engineering Walkthrough**.

### 📚 Additional Resources in this Repository:
- 📄 **Mentor Teaching Playbook:** `FinTrust_ML_Workflow_Week3_Mentor_Guide.docx`
- 📊 **Slide Presentation Deck:** `FinTrust_ML_Workflow_Week3_Intern_Presentation.pptx`
- 🛡️ **Test Report:** `docs/WEEK3_TEST_REPORT.md`
- 🏛️ **Architecture Reference:** `docs/WEEK3_DOCUMENTATION.md`